<a href="https://colab.research.google.com/github/SRIKRISHNA1407627/AI_LAB_1BM24CS151/blob/main/WEEK07_UNIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re

class Term:
    def apply_subst(self, subst): raise NotImplementedError

class Variable(Term):
    def __init__(self, name): self.name = name
    def apply_subst(self, subst):
        if self.name in subst: return subst[self.name].apply_subst(subst)
        return self
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Variable) and self.name == other.name
    def __hash__(self): return hash(self.name)

class Constant(Term):
    def __init__(self, name): self.name = name
    def apply_subst(self, subst): return self
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Constant) and self.name == other.name

class Function(Term):
    def __init__(self, name, args):
        self.name = name
        self.args = list(args)
    def apply_subst(self, subst):
        return Function(self.name, [arg.apply_subst(subst) for arg in self.args])
    def __repr__(self):
        # If it's our top-level container, print it cleanly without the wrapper name
        if self.name == "__TUPLE__":
            return ", ".join(map(str, self.args))
        return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, other):
        return isinstance(other, Function) and self.name == other.name and self.args == other.args


def occurs_check(var, term, subst):
    if var == term: return True
    if isinstance(term, Variable):
        if term.name in subst: return occurs_check(var, subst[term.name], subst)
        return False
    if isinstance(term, Function): return any(occurs_check(var, arg, subst) for arg in term.args)
    return False


def unify_var(var, term, subst):
    if var == term: return subst
    if var.name in subst: return unify(subst[var.name], term, subst)
    if isinstance(term, Variable) and term.name in subst: return unify(var, subst[term.name], subst)
    if occurs_check(var, term, subst): return None

    new_subst = subst.copy()
    new_subst[var.name] = term
    return new_subst


def unify(x, y, subst=None):
    if subst is None: subst = {}
    if subst is None: return None

    # Always resolve the terms against current substitutions before matching
    if isinstance(x, Term): x = x.apply_subst(subst)
    if isinstance(y, Term): y = y.apply_subst(subst)

    if x == y: return subst
    if isinstance(x, Variable): return unify_var(x, y, subst)
    if isinstance(y, Variable): return unify_var(y, x, subst)

    if isinstance(x, Function) and isinstance(y, Function):
        if x.name != y.name or len(x.args) != len(y.args): return None

        current_subst = subst
        for arg_x, arg_y in zip(x.args, y.args):
            current_subst = unify(arg_x, arg_y, current_subst)
            if current_subst is None: return None
        return current_subst
    return None

# =====================================================================
# Intelligent Sequence & Term Parser
# =====================================================================
def split_arguments(text):
    """Splits a comma-separated string, respecting balanced parentheses."""
    args = []
    bracket_count = 0
    current_arg = []
    for char in text:
        if char == ',' and bracket_count == 0:
            args.append("".join(current_arg).strip())
            current_arg = []
        else:
            if char == '(': bracket_count += 1
            elif char == ')': bracket_count -= 1
            current_arg.append(char)
    if current_arg:
        args.append("".join(current_arg).strip())
    return args

def parse_single_term(text):
    text = text.strip()
    match = re.match(r'^([A-Za-z0-9_]+)\s*\((.*)\)$', text)
    if match:
        name = match.group(1)
        args_str = match.group(2)
        args = [parse_single_term(arg) for arg in split_arguments(args_str)]
        return Function(name, args)

    if text.islower():
        return Variable(text)
    else:
        return Constant(text)

def parse_input(text):
    """Parses single terms OR comma-separated lists of terms."""
    parts = split_arguments(text.strip())
    if len(parts) > 1:
        # Wrap sequence inside a dummy Function container to evaluate sequentially
        return Function("__TUPLE__", [parse_single_term(p) for p in parts])
    return parse_single_term(parts[0])

# =====================================================================
# Execution Loop
# =====================================================================
def run_repl():
    print("=======================================================")
    print("   First-Order Logic (FOL) Unification Engine Shell   ")
    print("=======================================================")

    while True:
        try:
            expr1_raw = input("Enter first term  > ").strip()
            if expr1_raw.lower() in ['exit', 'quit']: break
            if not expr1_raw: continue
            term1 = parse_input(expr1_raw)

            expr2_raw = input("Enter second term > ").strip()
            if expr2_raw.lower() in ['exit', 'quit']: break
            if not expr2_raw: continue
            term2 = parse_input(expr2_raw)

            print(f"\nUnifying structural trees:\n  T1: {term1}\n  T2: {term2}")
            result = unify(term1, term2)

            if result is None:
                print("Result: ❌ Unification Failed! (x cannot bind to conflicting items)\n")
            elif not result:
                print("Result: ✅ Unification Succeeded! (Identity Match)\n")
            else:
                print("Result: ✅ Most General Unifier (MGU):")
                for var, mapping in result.items():
                    print(f"  {var} ➔ {mapping}")
                print()

        except Exception as e:
            print(f"Error: {e}\n")

if __name__ == "__main__":
    run_repl()


   First-Order Logic (FOL) Unification Engine Shell   
Enter first term  > y,G(A,B)
Enter second term > S,G(x,x)

Unifying structural trees:
  T1: y, G(A, B)
  T2: S, G(x, x)
Result: ❌ Unification Failed! (x cannot bind to conflicting items)

Enter first term  > A,B,C
Enter second term > x,y,z

Unifying structural trees:
  T1: A, B, C
  T2: x, y, z
Result: ✅ Most General Unifier (MGU):
  x ➔ A
  y ➔ B
  z ➔ C

Enter first term  > exit
